# Titanic Data Analysis (Work in Progress)

## 1. Data Loading

In [273]:
import pandas as pd
df_original = pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
df = df_original.copy()
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [274]:
# Dataset was checked and missing values were identified
# Dataset contains missing values in the variables Age, Cabin and Embarked
df.info()
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

## 2. Data Cleaning

In [275]:
# Cabin variable was removed due to a high proportion of missing values (>50%)
# and it is not very informative in its current form
df = df.drop('Cabin', axis=1)

In [276]:
# Missing values in Age variable were imputed using the median
df['Age'] = df['Age'].fillna(df['Age'].median())
df.isnull().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       2
dtype: int64

In [277]:
# Outliers in the Age variable were identified using IQR method
Q1 = df['Age'].quantile(0.25)
Q3 = df['Age'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = df[(df['Age']<lower) | (df['Age']>upper)]
outliers

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.00,3,1,349909,21.0750,S
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.00,0,0,113783,26.5500,S
15,16,1,2,"Hewlett, Mrs. (Mary D Kingcome)",female,55.00,0,0,248706,16.0000,S
16,17,0,3,"Rice, Master. Eugene",male,2.00,4,1,382652,29.1250,Q
33,34,0,2,"Wheadon, Mr. Edward H",male,66.00,0,0,C.A. 24579,10.5000,S
...,...,...,...,...,...,...,...,...,...,...,...
827,828,1,2,"Mallet, Master. Andre",male,1.00,0,2,S.C./PARIS 2079,37.0042,C
829,830,1,1,"Stone, Mrs. George Nelson (Martha Evelyn)",female,62.00,0,0,113572,80.0000,NaN
831,832,1,2,"Richards, Master. George Sibley",male,0.83,1,1,29106,18.7500,S
851,852,0,3,"Svensson, Mr. Johan",male,74.00,0,0,347060,7.7750,S


According to the IQR method, some age values that appear to be outliers are not data errors but actual passenger ages. For this reason, they have not been deleted or transformed.

In [278]:
# Missing values in the Embarked variable were imputed using the mode
df['Embarked'] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df.isna().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

## 3. Feature Engineering

In [279]:
# Age was grouped into life-stage categories to better interpret survival patterns
df['Age_Group'] = pd.cut(df['Age'], bins=[0,12,18,30,50,80], labels=['0-12', '12-18', '18-30', '30-50', '50-80'])
df.groupby('Age_Group', observed=True)['Survived'].mean()

Age_Group
0-12     0.579710
12-18    0.428571
18-30    0.331096
30-50    0.423237
50-80    0.343750
Name: Survived, dtype: float64

Survival rates vary by age group; the highest rates are observed among children (ages 0–12), 
while the lowest rates are observed among young adults (ages 18–30). 
This indicates that age is a significant factor in survival; 
however, the underlying reasons require a more detailed examination in conjunction with additional variables such as gender and passenger class.

In [280]:
# Survival rates were analyzed by Sex group
df.groupby('Sex')['Survived'].mean().round(2)

Sex
female    0.74
male      0.19
Name: Survived, dtype: float64

While the survival rate for women is approximately 74%, that for men is approximately 19%. This indicates that gender plays a significant role in survival rates.

In [281]:
# Survival rates were analyzed by Pclass group
df.groupby('Pclass')['Survived'].mean()

Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64

It appears that the survival rate is higher among the higher classes. This suggests that class plays a role in survival rates.